In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_reg_deep import preprocess, VN30, TARGETS
from models.regression.lstm import LongShortTermMemory as LSTM

In [5]:
model = LSTM(
    n_features=4,    # [open, high, low, close]
    n_layers=1,
    hidden_dim=64,
    fc_dim=32,
    output_dim=4,    # dự báo [open, high, low, close]
    dropout=0.3
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)
criterion = nn.SmoothL1Loss()

In [11]:
symbol = "ACB"
train_loader, valid_loader, test_loader, scaler  = preprocess(symbol, mode='lstm')

In [12]:
best_val_loss = float('inf')
n_epochs = 200

for epoch in range(1, n_epochs + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch, y_batch
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    # --- validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch, y_batch
            preds = model(X_batch)
            val_loss += criterion(preds, y_batch).item() * X_batch.size(0)
    val_loss /= len(valid_loader.dataset)

    scheduler.step()

    # --- checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'checkpoints_lstm/lstm_{symbol}.pth')

    if epoch % 10 == 0 or epoch == n_epochs:
        print(f"Epoch {epoch:3d}/{n_epochs}: "
              f"Train Loss = {train_loss:.6f}, "
              f"Valid Loss = {val_loss:.6f}, "
              f"Best Val Loss = {best_val_loss:.6f}, "
			  f"LR = {optimizer.param_groups[0]['lr']:.6f}")

Epoch  10/200: Train Loss = 0.023280, Valid Loss = 0.007049, Best Val Loss = 0.002091, LR = 0.000800
Epoch  20/200: Train Loss = 0.021922, Valid Loss = 0.002604, Best Val Loss = 0.001586, LR = 0.000640
Epoch  30/200: Train Loss = 0.018189, Valid Loss = 0.005923, Best Val Loss = 0.001283, LR = 0.000512
Epoch  40/200: Train Loss = 0.019170, Valid Loss = 0.001692, Best Val Loss = 0.001097, LR = 0.000410
Epoch  50/200: Train Loss = 0.018312, Valid Loss = 0.001113, Best Val Loss = 0.001097, LR = 0.000328
Epoch  60/200: Train Loss = 0.018068, Valid Loss = 0.001412, Best Val Loss = 0.001097, LR = 0.000262
Epoch  70/200: Train Loss = 0.019052, Valid Loss = 0.001628, Best Val Loss = 0.001097, LR = 0.000210
Epoch  80/200: Train Loss = 0.016214, Valid Loss = 0.001440, Best Val Loss = 0.001097, LR = 0.000168
Epoch  90/200: Train Loss = 0.016575, Valid Loss = 0.001575, Best Val Loss = 0.001097, LR = 0.000134
Epoch 100/200: Train Loss = 0.016803, Valid Loss = 0.002429, Best Val Loss = 0.001035, LR =

In [13]:
def eval(symbol: str):
    model = LSTM()
    _, _, test_loader, scaler = preprocess(symbol, 'lstm')
    model.load_state_dict(torch.load(f'checkpoints_lstm/lstm_{symbol}.pth', map_location='cpu'))
    model.eval()

    # Thu thập dự đoán và nhãn
    all_preds   = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch
            preds = model(X_batch).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y_batch.numpy())

    all_preds   = np.vstack(all_preds)   # (n_samples, 5)
    all_targets = np.vstack(all_targets)

    # Inverse scaling
    all_preds_inv   = scaler.inverse_transform(all_preds)
    all_targets_inv = scaler.inverse_transform(all_targets)

    # Tính metrics
    r2   = r2_score(all_targets_inv, all_preds_inv, multioutput='uniform_average')
    mape = mean_absolute_percentage_error(all_targets_inv, all_preds_inv) * 100
    
    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")
    return r2, mape 

In [14]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    r2, mape = eval(symbol)
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R^2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R^2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R^2: 0.7599, MAPE: 2.2795
Symbol: BCM, R^2: 0.9525, MAPE: 1.4179
Symbol: BID, R^2: 0.6899, MAPE: 2.3997
Symbol: BVH, R^2: 0.9799, MAPE: 1.1004
Symbol: CTG, R^2: 0.9558, MAPE: 1.3876
Symbol: FPT, R^2: 0.4188, MAPE: 9.1473
Symbol: GAS, R^2: 0.9602, MAPE: 0.7823
Symbol: GVR, R^2: 0.9424, MAPE: 2.5486
Symbol: HDB, R^2: 0.9023, MAPE: 2.4866
Symbol: HPG, R^2: 0.8843, MAPE: 1.1703
Symbol: LPB, R^2: 0.5012, MAPE: 12.1920
Symbol: MBB, R^2: 0.9164, MAPE: 1.6649
Symbol: MSN, R^2: 0.9503, MAPE: 1.1913
Symbol: MWG, R^2: 0.9784, MAPE: 1.4266
Symbol: PLX, R^2: 0.9837, MAPE: 1.0383
Symbol: SAB, R^2: 0.7855, MAPE: 2.0416
Symbol: SHB, R^2: 0.9597, MAPE: 1.1071
Symbol: SSB, R^2: 0.9171, MAPE: 1.5906
Symbol: SSI, R^2: 0.9303, MAPE: 1.2177
Symbol: STB, R^2: 0.9780, MAPE: 1.1786
Symbol: TCB, R^2: 0.9541, MAPE: 1.9643
Symbol: TPB, R^2: 0.9428, MAPE: 1.2918
Symbol: VCB, R^2: 0.8599, MAPE: 0.8501
Symbol: VHM, R^2: 0.9474, MAPE: 1.8607
Symbol: VIB, R^2: 0.8992, MAPE: 1.2798
Symbol: VIC, R^2: 0.9465